In [ ]:
from fmsae_repro import DATA, RESULTS, LEGACY_ANALYSIS
import ezslide
import lazyslide as zs
import mesoslide as ms

from matplotlib.colors import ListedColormap, LinearSegmentedColormap
# transparent_to_green_cmap = ListedColormap([(0, 0, 0, 0), (0, 1, 0, 0.5)])
transparent_to_green_cmap = LinearSegmentedColormap.from_list(
    'transparent_to_green', [(1, 1, 1, 0), (0, 1, 0, 1)], N=256
)
transparent_to_red_cmap = LinearSegmentedColormap.from_list(
    'transparent_to_red', [(1, 1, 1, 0.5), (1, 0, 0, 1)], N=256
)   

In [ ]:
wsi = ezslide.read_wsi(f'{DATA}/LSP24521.zarr', 
                       attach_image=True)
wsi

In [ ]:
import numpy as np
table = wsi.tables["tiles_table"]
col = "UNI_SAE_47380"
table.obs[col] = np.asarray(table[:, col].X.todense()).ravel()
zs.pl.tiles(wsi, feature_key="tiles_table", color=col, tissue_id="all")


In [ ]:
zs.pl.tiles(
    wsi,
    feature_key='tiles_table',
    tissue_id="all",
    color="UNI_SAE_47380",
    cmap=transparent_to_green_cmap,
)

In [ ]:
_ = ms.plotting.plot_feature_map(wsi, 'UNI_SAE_59985', figsize=(6,6))

In [ ]:
wsi.tables["tiles_table"][:, "UNI_SAE_59985"].X.max()

In [ ]:
import pandas as pd
slide_ids = []
slide_stores = []
lsp_id_start = 24521
for i in range(0, 40):
    lsp_id = lsp_id_start + i * 3
    slide_ids.append(f"LSP{lsp_id}")
    slide_stores.append(f"data/LSP{lsp_id}.zarr")
manifest = pd.DataFrame({"slide_id": slide_ids, "store": slide_stores})
manifest.head()

In [ ]:
manifest.to_csv(f"{DATA}/manifest.csv", index=False)

In [ ]:
slides = ezslide.open_slides(manifest, attach_images=False)

In [ ]:
patches = ms.select_random_patches(wsi, 20)
patches

In [ ]:
ms.plotting.plot_patch_gallery(patches, slides=wsi, return_fig=True)

In [ ]:
patches = ms.select_top_patches(wsi, 'UNI_SAE_47380', 20)
# ms.plotting.plot_patch_gallery(patches, slides=wsi, return_fig=True)

In [ ]:
ms.plotting.plot_patch_gallery(patches, slides=wsi, 
                               filename_prefix='UNI_SAE_47380',
                               output_dir=f'{RESULTS}/figures/SAE1/feature_reports', return_fig=False)

In [ ]:
import zarr
from tqdm import tqdm
for i in tqdm(range(40)):
    lsp_id = lsp_id_start + i * 3
    zarr.consolidate_metadata(f'{DATA}/LSP{lsp_id}.zarr')


In [ ]:
from tqdm import tqdm
lsp_id_start = 24521

slides = []
for i in tqdm(range(40)):
    lsp_id = lsp_id_start + i * 3
    slides.append(ezslide.read_wsi(f'{DATA}/LSP{lsp_id}.zarr'))

In [ ]:
%reload_ext autoreload
%autoreload 2

# import mesoslide as ms
# from mesoslide.plotting import create_feature_report

In [ ]:
import mesoslide; print(mesoslide.__file__)

In [ ]:
from mesoslide.plotting import create_feature_report
ms.plotting._feature_reports.create_feature_report(
    slides=slides,
    feature_name='UNI_SAE_46327',
    fill_alpha=1.0,
    top_fraction=0.1,
    nrows=2,
    output_path=f'{RESULTS}/figures/SAE1/feature_reports/UNI_SAE_59985.pdf',
)

In [ ]:
import itertools

my_dict = {
    "a": 1,
    "b": 2,
    "c": 3,
    "d": 4,
    "e": 5,
    "f": 6
}

# Start at index 1, stop at index 5, skip every 2nd item (step = 2)
skipped_dict = dict(itertools.islice(my_dict.items(), None, None, 2))

print(skipped_dict)
# Output: {'b': 2, 'd': 4}


In [ ]:
patches = ms.select_top_patches(slides, 'UNI_SAE_47380', 20, 
                                take_every=200)
patches.obs

In [ ]:
ms.plotting.plot_patch_gallery(patches, slides, return_fig=True)

### Test SAE feature selection

In [ ]:
from mesoslide.tools import SAEFeatureSelector

selector = SAEFeatureSelector(
    pct_threshold=0.002,
    max_score_threshold=0.1,
    n_chunks=40
)
selector.compute_activation_stats(slides, 
                                  feature_prefix='UNI_SAE', 
                                  num_features=65536)
g = selector.plot_feature_selection()
# g.savefig(f"{RESULTS}/figures/SAE_feature_selection.pdf", 
#           dpi=300, bbox_inches='tight', pad_inches=0, transparent=False, facecolor='white')
# plt.close(g.fig)

In [ ]:
highly_active_features_idx = selector.get_selected_indices()
len(highly_active_features_idx)

In [ ]:
# Cluster
from mesoslide.tools.sae import SAEFeatureClusterer

clusterer = SAEFeatureClusterer(high_activation_threshold=0.5)
clusterer.compute_iou(slides, feature_prefix='UNI_SAE',
                      feature_indices=highly_active_features_idx)
clusterer.cluster(threshold=25, criterion='maxclust')

In [ ]:
# Plot heatmap
g = clusterer.plot_heatmap()
# g.savefig(f"{LEGACY_ANALYSIS}/SAE_paper/figures/Supp_Fig_4/feature_pairwise_iou_full.pdf", dpi=300)

In [ ]:
exemplar_patches = ms.select_exemplar_patches(slides, [f'UNI_SAE_{idx}' for idx in highly_active_features_idx])
exemplar_patches

In [ ]:
from mesoslide.plotting._patch_gallery import plot_feature_gallery
from mesoslide.preprocessing._extract_patches import extract_patches
import distinctipy
from matplotlib.colors import ListedColormap

# ── configure ──────────────────────────────────────────────────────────────────
# groups_to_plot = {3, 4, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25}
groups_to_plot = list(range(1, 26))
colors = distinctipy.get_colors(25, rng=43)
cmap = ListedColormap(colors)

# ── extract pixel patches once, keyed by global feature index ──────────────────
exemplar_images = extract_patches(exemplar_patches, slides, channel_first=False, progress_bar=True)
feature_to_image = {
    int(name.rsplit('_', 1)[-1]): img
    for name, img in zip(exemplar_patches.obs['_feature_name'], exemplar_images)
}

# ── filter by group ────────────────────────────────────────────────────────────
ordered_idx      = clusterer.feature_indices_[clusterer.reordered_idx_]
reordered_groups = clusterer.reordered_clusters_.tolist()

mask      = [i for i, g in enumerate(reordered_groups) if g in groups_to_plot]
patches   = [feature_to_image[ordered_idx[i]] for i in mask]
group_ids = [reordered_groups[i] for i in mask]
labels    = None  # or [str(ordered_idx[i]) for i in mask] if you want labels

# ── plot ───────────────────────────────────────────────────────────────────────
fig, axs = plot_feature_gallery(
    images=patches,
    group_ids=group_ids,
    labels=labels,
    n_cols=10,
    patch_size=1.0,
    border_extend=0.05,
    border_alpha=1.0,
    cmap=cmap,
    fontsize=6,
)
# fig.savefig(f'{LEGACY_ANALYSIS}/SAE_paper/figures/Fig_3/exemplar_patches_cellular_features.png', dpi=300)

In [ ]:
ms.plotting.plot_patch_gallery(
    patches=exemplar_patches,
    slides=slides,
    output_path=f'{RESULTS}/figures/SAE1/exemplar_patches',
)
